In [ ]:
%%bash
if [ -d "./cifar10_data/cifar-10-batches-py" ] && [ -f "./cifar10_data/cifar-10-batches-py/batches.meta" ]; then
    echo "Dataset already exists, skipping download."
else
    echo "Dataset not found, downloading..."
    kaggle datasets download -d pankrzysiu/cifar10-python --force
    unzip -q cifar10-python.zip -d ./cifar10_data
fi

In [ ]:
import pickle
import numpy as np
import os


def unpickle(file):
    with open(file, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')
    return data_dict


data_dir = './cifar10_data/cifar-10-batches-py/'

x_train_list = []
y_train_list = []

for i in range(1, 6):
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    batch = unpickle(batch_path)
    x_train_list.append(batch[b'data'])
    y_train_list.append(batch[b'labels'])

x_train = np.concatenate(x_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

test_batch = unpickle(os.path.join(data_dir, 'test_batch'))
x_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])

meta_batch = unpickle(os.path.join(data_dir, 'batches.meta'))
classes = [b.decode('utf-8') for b in meta_batch[b'label_names']]

print(f"Training features shape: {x_train.shape}")
print(f"Training labels shape:   {y_train.shape}")
print(f"Testing features shape:  {x_test.shape}")
print(f"Dataset classes:         {classes}")

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

x_train = x_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
x_test = x_test.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

plt.figure(figsize=(10, 2))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(x_train[i])
    plt.title(classes[y_train[i]])
    plt.axis('off')
plt.show()

### CIFAR-10 Dataset Structure
- **Dimensions**: The dataset features images of size 32x32 pixels with 3 color channels (RGB).
- **Classes**: There are 10 classes in total: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck.
- **Samples**: 50,000 training images and 10,000 testing images.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomFlip("horizontal")
])

augmented_images = [data_augmentation(np.expand_dims(x_train[0], 0), training=True)[0] for _ in range(4)]

plt.figure(figsize=(6, 6))
plt.suptitle("Augmented Samples")
for i in range(4):
    plt.subplot(2, 2, i+1)
    plt.imshow(augmented_images[i].numpy())
    plt.axis('off')
plt.show()

### Importance of Data Augmentation
Data augmentation artificially expands the training dataset by applying random transformations like rotations, shifts, and flips to the images. This helps the CNN generalize better, prevents overfitting, and ensures the model is robust to variations in object position, orientation, and scale in real-world scenarios.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()

### CNN Architecture Design Choices
- **Data Augmentation Layer**: Integrated at the beginning of the model to apply random transformations on the fly during training.
- **Convolutional Layers (Conv2D)**: Extract local spatial features (edges, textures) with increasing levels of abstraction using ReLU activations for non-linearity.
- **Pooling Layers (MaxPooling2D)**: Reduce the spatial dimensions (width and height), retaining the most significant features and reducing computational overhead.
- **Fully Connected Layers (Dense)**: Map the high-level features extracted by convolutional blocks to the 10 target classes using a Softmax activation to output probabilities.

In [ ]:
x_train_sub = x_train[:5000]
y_train_sub = y_train[:5000]
x_val_sub = x_train[5000:6000]
y_val_sub = y_train[5000:6000]

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(x_train_sub, y_train_sub, epochs=10, batch_size=64, validation_data=(x_val_sub, y_val_sub))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix
import seaborn as sns

test_subset_size = 1000
x_test_sub = x_test[:test_subset_size]
y_test_sub = y_test[:test_subset_size]

y_pred_probs = model.predict(x_test_sub)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = accuracy_score(y_test_sub, y_pred)
precision = precision_score(y_test_sub, y_pred, average='macro')
recall = recall_score(y_test_sub, y_pred, average='macro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}\n")

print("Classification Report:")
print(classification_report(y_test_sub, y_pred, target_names=classes))

cm = confusion_matrix(y_test_sub, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
plt.suptitle("Sample Predictions")
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(x_test_sub[i])
    pred_label = classes[y_pred[i]]
    true_label = classes[y_test_sub[i]]
    plt.title(f"Pred: {pred_label}\nTrue: {true_label}", color='green' if y_pred[i] == y_test_sub[i] else 'red')
    plt.axis('off')
plt.show()